In [ ]:
import h5py
import cv2
import numpy as np
from pathlib import Path
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

In [ ]:
def setup_folders():
    folders = ['final_result']

    for f in folders:
        Path(f).mkdir(parents=True, exist_ok=True)

    print(f"Folder has been created: {folders}")

In [ ]:
def post_processing(h5_path, output_folder, min_distance, cell_prob_threshold, min_cell_size, buffer):
    h5_path = Path(h5_path)
    output_folder = Path(output_folder)

    h5_filename = h5_path.stem
    output_path = output_folder / h5_filename

    output_path.mkdir(parents=True, exist_ok=True) #parents=True隨路徑遞歸建立目錄 #exist_ok=True存在即跳過

    report_lines = []

    with h5py.File(h5_path, 'r') as f:
        for img_name in f.keys(): #f.keys()釣出f裡所有group名組成list
            print(f"processing image: {img_name}")

            group = f[img_name] #a[目標][指定讀取範圍, 進入記憶體]
            raw_img = group['raw_image'][:]
            prob_map = group['probability_map'][:]
            h, w = prob_map.shape #raw_img.shape = (H, W, 3), prob_map.shape = (H, W)

            thresh = prob_map >= cell_prob_threshold
            distance = ndi.distance_transform_edt(thresh)
                #Euclidean Distance Transform, EDT #必須輸入boolean matrix, False=背景像素, True=前景像素
                #算每一個True與最近的False的EDT距離, 存成matrix
            dist_smoothed = gaussian_filter(distance, sigma=30)
            coords = peak_local_max(dist_smoothed, min_distance=min_distance, labels=thresh)
                #peak_local_max()在一維或多維矩陣中, 尋找局部區域內的極大值peaks
                #peak_local_max(目標矩陣, 最短距離threshold, 區域限制) #最短距離threshold 加大減少沾黏, 縮小減少過度計算
                #區域限制 根據與目標矩陣同大小的boolean matrix, True即為範圍
            mask = np.zeros(distance.shape, dtype=bool)
            mask[tuple(coords.T)] = True
                #將細胞中心座標寫成boolean matrix
                #mask[] numpy多維索引賦值, 需求格式為(array([y1, y2, y3]), array([x1, x2, x3]))
                #coords原格式[[y1, x1], [y2, x2], ...] >>> .T後[[y1, y2, y3, ...], [x1, x2, x3, ...]]
                #tuple()元祖化(array([y1, y2, y3, ...]), array([x1, x2, x3, ...]))
            markers, _ = ndi.label(mask) #a, b = ndi.label(矩陣) 將矩陣中兩個相連的非0(or True)元素視為同一元素, 輸出a為元素矩陣map(標記編號), 輸出b為元素數量
            labels = watershed(-distance, markers, mask=thresh)
                #分水領算法Marker-controlled Watershed Algorithm #watershed(地形圖, 谷地中心, 邊界)
                #地形圖 distance加負號為數字做反轉, 原正數時中心最高, 需要中心最低
                #谷地中心 標記帶有編號的獨立中心 #邊界 用boolean matrix的True標記邊界

            for label_id in np.unique(labels):
                if label_id == 0: continue
                cell_area = np.sum(labels == label_id)

                if cell_area < min_cell_size:
                    labels[labels == label_id] = 0

            complete_count = 0
            incomplete_count = 0
            overlay_img = raw_img.copy()

            for label_id in np.unique(labels): #np.unique()找出矩陣中所有出現過的不重複數值, 並由小到大排列, 格式[0, 1, 2, 3, ..., N]
                if label_id == 0: continue
                cell_mask = (labels == label_id) #每一次for loop為該編號細胞生成一張boolean matrix
                y_coords, x_coords = np.where(cell_mask) #np.where()用y, x回傳所有非0數值的座標, 格式y_coords = array([]), x_coords = array([])

                on_border = (np.any(y_coords <= buffer) or np.any(y_coords >= (h - buffer)) or
                             np.any(x_coords <= buffer) or np.any(x_coords >= (w - buffer)))

                if on_border:
                    incomplete_count += 1
                    labels[labels == label_id] = 0
                    #color = (0, 0, 255) #(B, G, R)
                else:
                    complete_count += 1
                    color = (0, 255, 0)
                    contours, _ = cv2.findContours(cell_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                        #cv2.findContours(目標矩陣, 模式, 儲存方式) #目標矩陣 需要是uint8格式 #模式cv2.RETR_EXTERNAL 緊抓取輪廓
                        #儲存方式cv2.CHAIN_APPROX_SIMPLE 遇直線僅記錄頭尾 #contours 輸出格式array([[[x1, y1]], [[x2, y2]]], ..., dtype=int32)
                    cv2.drawContours(overlay_img, contours, -1, color, 2)
                        #cv2.drawContours(畫布, 座標, 指定繪製目標contourIdx, 顏色, 線條寬度) 繪製輪廓
                        #contourIdx=-1 參數-1表示繪製全部

            thresh = (labels > 0)
            lower_bound = cell_prob_threshold
            upper_bound = np.max(prob_map)

            if upper_bound > lower_bound:
                norm_prob = np.clip((prob_map - lower_bound) / (upper_bound - lower_bound), 0, 1)
            else:
                norm_prob = np.zeros_like(prob_map)

            heatmap = cv2.applyColorMap((norm_prob * 255).astype(np.uint8), cv2.COLORMAP_MAGMA)
            #cv2.applyColorMap(8-bit亮度圖矩陣, 選擇填色模式) 填色工具 #(norm_prob * 255) 映射成8 bit的0到255亮度

            mask_3ch = np.repeat(thresh[:, :, None], 3, axis=2)
            #np.repeat(目標矩陣, 重複次數, 目標維度) 複製維度 #目標維度數字*重複次數=結果, ex.np.repeat((1, 2), 3, axis=1) >>> shape = (1, 6)
            heatmap_overlay = np.where(mask_3ch, heatmap, raw_img) #input shape = (H, W, 3) #np.where(boolean矩陣, true時填入, false時填入)

            img_stem = Path(img_name).stem
            cv2.imwrite(str(output_path / f"{img_stem}_counter.png"), overlay_img)
            cv2.imwrite(str(output_path / f"{img_stem}_heatmap.png"), heatmap_overlay)
            cv2.imwrite(str(output_path / f"{img_stem}_origin.png"), raw_img)

            total = complete_count + incomplete_count
            report_lines.append(f"{img_name}: total {complete_count} cells")

    report_path = output_path / f"{h5_filename}_report.txt"
    with report_path.open("w", encoding="utf-8") as txt_file:
        txt_file.write("\n".join(report_lines))

In [ ]:
def batch_post_process(input_folder, output_path, min_distance, cell_prob_threshold, min_cell_size, buffer):
    input_path = Path(input_folder)
    output_path = Path(output_path)

    h5_files = list(input_path.glob("*.h5"))

    if not h5_files:
        print(f"file not exist")
        return

    print(f"total files: {len(h5_files)}")

    for h5_file in h5_files:
        print(f"processing file: {h5_file.name}")
        post_processing(h5_file, output_path, min_distance=min_distance,
                        cell_prob_threshold=cell_prob_threshold, min_cell_size=min_cell_size, buffer=buffer)

In [ ]:
if __name__ == "__main__":
    setup_folders()
    
    input_folder = "prediction_results"
    output_path = "final_result"

    batch_post_process(input_folder, output_path, min_distance=90, cell_prob_threshold=0.3, min_cell_size=6400, buffer=3)
    print("complete.")